In [2]:
import cv2
from ultralytics import YOLO
import numpy as np
from pathlib import Path

#### Crop Boxes Helper

In [3]:
def crop_from_bbox(img, bbox, pad=0):
    x1, y1, x2, y2 = bbox
    h, w = img.shape[:2]

    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(w, x2 + pad)
    y2 = min(h, y2 + pad)

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return crop

#### Vehicle and Plate Models

In [4]:
vehicle_model   = YOLO("yolov8n.pt")
class_list = vehicle_model.names
VEHICLE_CLASSES = {
    'car',
    'motorcycle',
    'bus',
    'truck'   # includes vans & lorries in COCO
}
plate_model     = YOLO("/home/joshu/LPR-proj/plateDetectModel/best.pt")

#### Read in image

In [5]:
image_path = "/home/joshu/LPR-proj/data/LPRdatasets/plateImages/008e36a5a0623445_jpg.rf.c0df6282d2fe4484c4c6de82acbb2ab5.jpg"
image = cv2.imread(image_path)

if image is None: 
    raise FileNotFoundError(image_path)

#### Vehicle Detection

In [6]:
vehicle_results = vehicle_model.predict(image, conf = 0.4, device="cpu", verbose = False)
detections = vehicle_results[0].boxes.data.detach().cpu().numpy()

vehicle_crops = []

for det in detections:
    x1, y1, x2, y2 = map(int, det[:4])
    confidence = float(det[4])
    class_id = int(det[5])
    class_name = class_list[class_id]

    if class_name in VEHICLE_CLASSES:
        v_crop = crop_from_bbox(image, (x1, y1, x2, y2), pad=5)
        if v_crop is None:
            continue

        vehicle_crops.append({
            "class": class_name,
            "confidence": confidence,
            "bbox": {x1, y1, x2, y2},
            "crop": v_crop
        })

#### Plate Detection

In [7]:
plate_crops = []

for vi, v in enumerate(vehicle_crops):
    v_img = v["crop"]

    plate_results = plate_model.predict(v_img, conf=0.25, iou = 0.45, device ="cpu", verbose = False)
    r = plate_results[0]

    if r.boxes is None or len(r.boxes) == 0:
        continue

    xyxy = r.boxes.xyxy.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()

    #picks one plate per image, the largest
    areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    idx = int(np.argmax(areas))

    px1, py1, px2, py2 = xyxy[idx].astype(int)
    pconf = float(confs[idx])

    p_crop = crop_from_bbox(v_img, (px1, py1, px2, py2), pad=5)
    if p_crop is None:
        continue

    plate_crops.append({
        "vehicle_index": vi,
        "plate_bbox_in_vehicle": (px1, py1, px2, py2),
        "plate_conf": pconf,
        "crop": p_crop
    })

#### Save cropped images (change later)

In [8]:
out_dir = Path("/home/joshu/LPR-proj/outputs/plates")
out_dir.mkdir(parents=True, exist_ok=True)

for i, p in enumerate(plate_crops):
    out_path = out_dir / f"plate_{i}_veh{p['vehicle_index']}.jpg"
    cv2.imwrite(str(out_path), p["crop"])

print(f"Saved to: {out_dir}")

Saved to: /home/joshu/LPR-proj/outputs/plates


In [9]:
print(f"Vehicles found: {len(vehicle_crops)}")
print(f"Plates cropped: {len(plate_crops)}")

Vehicles found: 2
Plates cropped: 1
